# Phase 4 — RB Model: Tuned XGBoost with Walk-Forward Optuna Search

Trained only on RB's Phase 3-selected 4 features: `scarcity_z`, `vorp_delta_yoy`, `age`, `draft_pick_inverse`. Built `XGBRegressor`-first from the start (not `xgb.train()`/`DMatrix`), matching the interface QB's notebook (`06a_model_qb.ipynb`) was converted to, so the two positions stay consistent going forward. Testing whether tuning meaningfully closes the gap Phase 3's untuned model left against the naive baseline (0.9% MAE, +0.008 Spearman — the smallest lift of any position).

In [1]:
from datetime import datetime

print(f"Results as of {datetime.now().astimezone():%Y-%m-%d %H:%M %Z}, pulling live nflreadpy data -- "
      "rerunning this notebook will reflect any upstream corrections made to that data since.")

Results as of 2026-09-15 17:25 Central Daylight Time, pulling live nflreadpy data -- rerunning this notebook will reflect any upstream corrections made to that data since.


## Setup: load `features_df` and build RB's target rows

Only the 4 already-selected features are built here — not the full Phase 3 pipeline. RB's feature set doesn't include any of the receiving-usage trio, the OL run-blocking proxy, or any of the EPA columns (RFECV eliminated all of them for RB — see `05_feature_selection.ipynb`), so no play-by-play or team-stats pull is needed at all for this notebook.

In [2]:
import sys
from pathlib import Path

import nflreadpy as nfl
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error

optuna.logging.set_verbosity(optuna.logging.WARNING)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FEATURES = ["scarcity_z", "vorp_delta_yoy", "age", "draft_pick_inverse"]

vorp_labels = pd.read_parquet(REPO_ROOT / "data/processed/vorp_labels.parquet")
rb = vorp_labels[vorp_labels["position"] == "RB"][
    ["season", "player_id", "player_display_name", "vorp", "vorp_next"]
].copy()

# scarcity_z (within-position, within-season standardization -- same recipe as Phase 3)
season_position_stats = (
    vorp_labels.groupby(["season", "position"])["vorp"]
    .agg(position_mean_vorp="mean", position_std_vorp_that_season="std")
    .reset_index()
)
rb_stats = season_position_stats[season_position_stats["position"] == "RB"]
rb = rb.merge(rb_stats[["season", "position_mean_vorp", "position_std_vorp_that_season"]], on="season", how="left")
rb["scarcity_z"] = (rb["vorp"] - rb["position_mean_vorp"]) / rb["position_std_vorp_that_season"]
rb = rb.drop(columns=["position_mean_vorp", "position_std_vorp_that_season"])

# vorp_delta_yoy
prior = rb[["player_id", "season", "vorp"]].copy()
prior["season"] = prior["season"] + 1
prior = prior.rename(columns={"vorp": "vorp_last_season"})
rb = rb.merge(prior, on=["player_id", "season"], how="left")
rb["vorp_delta_yoy"] = rb["vorp"] - rb["vorp_last_season"]
rb = rb.drop(columns=["vorp_last_season"])

# age, draft_pick_inverse
players = nfl.load_players().to_pandas()
rb = rb.merge(players[["gsis_id", "birth_date", "draft_pick"]], left_on="player_id", right_on="gsis_id", how="left")
rb["birth_date"] = pd.to_datetime(rb["birth_date"])
season_start = pd.to_datetime(rb["season"].astype(str) + "-09-01")
rb["age"] = (season_start - rb["birth_date"]).dt.days / 365.25
rb["draft_pick_inverse"] = 1 / rb["draft_pick"]
rb = rb.drop(columns=["gsis_id", "birth_date", "draft_pick"])

rb = rb.dropna(subset=["vorp_next"]).reset_index(drop=True)
print(f"RB rows with a usable label: {len(rb)}, seasons {rb['season'].min()}-{rb['season'].max()}")
print(rb[FEATURES].isna().mean().rename("null_rate"))

C:\Users\viraj\OneDrive\Desktop\ML Trial\ML-Test\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RB rows with a usable label: 1823, seasons 2008-2024
scarcity_z            0.000000
vorp_delta_yoy        0.297312
age                   0.000000
draft_pick_inverse    0.277564
Name: null_rate, dtype: float64


## Walk-forward folds (same structure as `05_feature_selection.ipynb` / `06a_model_qb.ipynb`)

In [3]:
MIN_TRAIN_SEASONS = 9


def make_walk_forward_folds(df, min_train_seasons=MIN_TRAIN_SEASONS):
    seasons_sorted = sorted(df["season"].unique())
    folds = []
    for i in range(min_train_seasons, len(seasons_sorted)):
        train_seasons = sorted(seasons_sorted[:i])
        test_season = seasons_sorted[i]
        train_idx = df.index[df["season"].isin(train_seasons)].to_numpy()
        test_idx = df.index[df["season"] == test_season].to_numpy()
        if len(train_idx) > 0 and len(test_idx) > 0:
            folds.append((train_idx, test_idx, test_season, train_seasons))
    return folds


folds = make_walk_forward_folds(rb)
print(f"{len(folds)} walk-forward folds, test seasons: {[f[2] for f in folds]}")

8 walk-forward folds, test seasons: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


## Monotonic constraints: checked per feature, not assumed

Each of RB's 4 features checked individually for whether it has an unambiguous, confound-free "more is better/worse" relationship with next-season value:

| Feature | Constraint | Reasoning |
|---|---|---|
| `scarcity_z` | **+1** | Standing further above the field *this* season should never predict a *worse* expected standing next season, all else equal. Unambiguous — same reasoning as QB. |
| `vorp_delta_yoy` | **+1** | An improving trajectory is more plausible to continue than to reverse, especially once `scarcity_z` already anchors the current level. Same reasoning as QB. |
| `age` | **0 (none)** | Explicitly NOT monotonic — and more pronounced for RB than any other position: Phase 3's own feature-importance write-up already named RB's "shelf-life cliff" (production rises then falls hard, with the whole receiving-usage trio getting eliminated once `age` and `scarcity_z` are known). Forcing a single direction here would contradict the real shape of that curve. |
| `draft_pick_inverse` | **0 (none)** | Same real ambiguity as QB: a fixed pre-career proxy for opportunity/organizational investment, not a performance metric. Conditional on `scarcity_z`/`vorp_delta_yoy` already carrying realized performance, it isn't clean enough to force a direction — could be picking up sunk-cost bias as easily as true signal. Left unconstrained rather than guessed. |

**2 of 4 features get a constraint** (`scarcity_z`, `vorp_delta_yoy`, both `+1`); `age` and `draft_pick_inverse` are left free. Same deliberately-conservative rule as QB — a wrong constraint forces an incorrect shape, which is worse than no constraint at all.

In [4]:
MONOTONE_CONSTRAINTS = (1, 1, 0, 0)  # matches FEATURES order exactly
print(dict(zip(FEATURES, MONOTONE_CONSTRAINTS)))

{'scarcity_z': 1, 'vorp_delta_yoy': 1, 'age': 0, 'draft_pick_inverse': 0}


## Per-fold Optuna search (TPE sampler, pruning enabled)

For each outer walk-forward fold, the search space is `max_depth` (2-8), `min_child_weight` (1-10), `reg_lambda` (log-scale 0.1-10), `learning_rate` (log-scale 0.01-0.3), `subsample` (0.6-1.0). `n_estimators` is never tuned directly — each trial uses early stopping (up to 500 rounds, 20-round patience) to find its own iteration count.

`max_depth`'s range was widened from an earlier 2-4 band to 2-8, matching the same change made to `06a_model_qb.ipynb` after `05b_qb_feature_experiment.ipynb` found the shallow-tree cap was artificial for QB — checked here for RB too rather than assuming QB's result carries over.

**The search never touches the outer fold's held-out test season.** Within each outer fold's own training window, the *most recent* training season is carved out as an inner validation split (everything before it is inner-train) — Optuna's objective is scored purely on that inner split. Real mid-training pruning is wired in via a custom `xgboost.callback.TrainingCallback` that reports each boosting round's validation MAE back to Optuna's `MedianPruner`, not just a pass/fail after the fact.

Once a fold's best hyperparameters and iteration count are found (from the inner split alone), the model is refit on the *entire* outer-fold training window before being scored on the real, still-untouched held-out test season.

**Interface**: this notebook is `XGBRegressor`-first from the start — `XGBRegressor(..., callbacks=[OptunaPruningCallback(trial)]).fit(X_train, y_train, eval_set=[(X_valid, y_valid)])`, not `xgb.train()`/`DMatrix`. `XGBRegressor`'s eval sets are auto-named `validation_0`, `validation_1`, ..., so the pruning callback reads `evals_log["validation_0"]` rather than a caller-chosen name.

In [5]:
N_TRIALS = 40


class OptunaPruningCallback(xgb.callback.TrainingCallback):
    """Reports each boosting round's validation MAE back to Optuna so its
    pruner can stop a clearly-unpromising trial mid-training, not just
    compare finished trials against each other after the fact.

    Reads "validation_0" -- XGBRegressor's auto-generated eval_set name --
    not the "valid" key the native xgb.train(evals=[...]) API would use."""

    def __init__(self, trial):
        self.trial = trial

    def after_iteration(self, model, epoch, evals_log):
        score = evals_log["validation_0"]["mae"][-1]
        self.trial.report(score, step=epoch)
        if self.trial.should_prune():
            raise optuna.TrialPruned()
        return False


def run_optuna_for_fold(df, train_idx, monotone_constraints, n_trials=N_TRIALS, seed=42):
    train_seasons_sorted = sorted(df.loc[train_idx, "season"].unique())
    inner_valid_season = train_seasons_sorted[-1]
    inner_train_seasons = train_seasons_sorted[:-1]
    inner_train_idx = df.index[df["season"].isin(inner_train_seasons) & df.index.isin(train_idx)]
    inner_valid_idx = df.index[(df["season"] == inner_valid_season) & df.index.isin(train_idx)]

    X_train = df.loc[inner_train_idx, FEATURES]
    y_train = df.loc[inner_train_idx, "vorp_next"]
    X_valid = df.loc[inner_valid_idx, FEATURES]
    y_valid = df.loc[inner_valid_idx, "vorp_next"]

    def objective(trial):
        params = {
            "objective": "reg:squarederror", "eval_metric": "mae",
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10, log=True),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "monotone_constraints": monotone_constraints, "seed": seed,
            "n_estimators": 500, "early_stopping_rounds": 20,
            "callbacks": [OptunaPruningCallback(trial)],
        }
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
        trial.set_user_attr("best_iteration", model.best_iteration)
        return model.best_score

    study = optuna.create_study(
        direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study


def fold_gain_share(model):
    """Gain importance for one fold's refit booster, normalized to shares
    over FEATURES (0 for any feature the booster never split on) -- same
    definition Phase 3 used (mean gain share across walk-forward folds)."""
    scores = model.get_booster().get_score(importance_type="gain")
    raw = np.array([scores.get(f, 0.0) for f in FEATURES])
    total = raw.sum()
    return raw / total if total > 0 else raw


def run_all_folds(df, folds, monotone_constraints, label):
    fold_results, fold_best_params, fold_gain_shares = [], [], []
    for tr, te, test_season, train_seasons in folds:
        study = run_optuna_for_fold(df, tr, monotone_constraints)
        best_params = dict(study.best_params)
        best_iteration = study.best_trial.user_attrs["best_iteration"]

        X_train_full = df.loc[tr, FEATURES]
        y_train_full = df.loc[tr, "vorp_next"]
        X_test = df.loc[te, FEATURES]
        final_model = xgb.XGBRegressor(
            objective="reg:squarederror", monotone_constraints=monotone_constraints, seed=42,
            n_estimators=max(best_iteration, 1), **best_params,
        )
        final_model.fit(X_train_full, y_train_full)
        preds = final_model.predict(X_test)
        actual = df.loc[te, "vorp_next"]

        mae = mean_absolute_error(actual, preds)
        rmse = mean_squared_error(actual, preds) ** 0.5
        rho = spearmanr(actual, preds)[0] if len(actual) >= 2 and actual.nunique() > 1 else np.nan

        fold_results.append({"position": "RB", "test_season": test_season, "n_test": len(te), "mae": mae, "rmse": rmse, "spearman": rho})
        fold_best_params.append(best_params | {"n_estimators": best_iteration})
        fold_gain_shares.append(fold_gain_share(final_model))

    fold_df = pd.DataFrame(fold_results)
    mean_gain_df = pd.DataFrame({"feature": FEATURES, "mean_gain_share": np.mean(fold_gain_shares, axis=0)})
    print(f"[{label}] Mean MAE: {fold_df['mae'].mean():.2f}, Mean RMSE: {fold_df['rmse'].mean():.2f}, Mean Spearman: {fold_df['spearman'].mean():.3f}")
    return fold_df, pd.DataFrame(fold_best_params), mean_gain_df


fold_metrics_df, fold_params_df, gain_constrained_df = run_all_folds(rb, folds, MONOTONE_CONSTRAINTS, label="tuned + monotonic")
print(fold_metrics_df.to_string(index=False))

[tuned + monotonic] Mean MAE: 46.67, Mean RMSE: 60.84, Mean Spearman: 0.691
position  test_season  n_test       mae      rmse  spearman
      RB         2017     104 47.684329 62.566688  0.637728
      RB         2018     107 45.447622 58.585677  0.692603
      RB         2019     114 44.295220 58.916990  0.661446
      RB         2020     123 41.442945 54.690718  0.714509
      RB         2021     120 49.205505 62.002632  0.708168
      RB         2022     114 53.385461 68.159304  0.600672
      RB         2023     108 46.345914 58.398501  0.781678
      RB         2024     106 45.559806 63.403553  0.733776


## Diagnostic: does the monotonic constraint actually help?

Domain reasoning alone doesn't guarantee a constraint helps a small dataset — checked directly rather than assumed, the same way QB's notebook (and every other claim in this project) already has been.

In [6]:
NO_CONSTRAINTS = (0, 0, 0, 0)
fold_metrics_unconstrained_df, _, gain_unconstrained_df = run_all_folds(rb, folds, NO_CONSTRAINTS, label="tuned, no constraints")

[tuned, no constraints] Mean MAE: 47.90, Mean RMSE: 61.74, Mean Spearman: 0.691


**Finding, reported honestly, and the opposite direction from QB**: for RB, the monotonic constraints *help* on MAE — 46.67 MAE constrained vs. 47.90 MAE unconstrained, a real ~2.6% difference. Spearman, unlike the old 2-4 search, comes out essentially tied this time: 0.691 constrained vs. 0.691 unconstrained. Net read: constraining still wins clearly on the metric this project has treated as primary (MAE), now at effectively no ranking-quality cost at all (the earlier 2-4 search had shown a small Spearman cost to constraining; the wider search erases it). This is still the opposite of QB's notebook, where dropping the constraints won outright on both metrics — a real, position-specific difference worth keeping in mind rather than assuming one position's constraint tradeoff generalizes to the next: RB's two constrained features (`scarcity_z`, `vorp_delta_yoy`) apparently cost the model very little to shape correctly, unlike QB's `passing_epa`.

### `draft_pick_inverse` vs `age`: actual gain values, not just rank

RB's two non-`scarcity_z`/`vorp_delta_yoy` features — same comparison QB ran for its own two secondary features, using the same methodology as Phase 3 (mean gain share across the walk-forward folds' own refit models, not a single all-data model).

In [7]:
# Phase 3's original untuned mean gain shares (05_feature_selection.ipynb), for the same 2 features.
phase3_gain = {"draft_pick_inverse": 0.087252, "age": 0.062606}

focus = ["draft_pick_inverse", "age"]
rows = []
for f in focus:
    constrained_val = gain_constrained_df.set_index("feature").loc[f, "mean_gain_share"]
    unconstrained_val = gain_unconstrained_df.set_index("feature").loc[f, "mean_gain_share"]
    rows.append({
        "feature": f,
        "phase3_untuned": phase3_gain[f],
        "phase4_tuned_constrained": constrained_val,
        "phase4_tuned_unconstrained": unconstrained_val,
    })
focus_df = pd.DataFrame(rows)
print(focus_df.to_string(index=False))

print("\nGap (draft_pick_inverse - age), each run:")
for col in ["phase3_untuned", "phase4_tuned_constrained", "phase4_tuned_unconstrained"]:
    gap = focus_df.set_index("feature").loc["draft_pick_inverse", col] - focus_df.set_index("feature").loc["age", col]
    ratio = focus_df.set_index("feature").loc["draft_pick_inverse", col] / focus_df.set_index("feature").loc["age", col]
    print(f"  {col}: gap={gap:+.4f}, draft_pick_inverse is {ratio:.2f}x age")

print("\nFull gain tables (all 4 features), for context:")
print("\nConstrained (Phase 4 official):")
print(gain_constrained_df.sort_values("mean_gain_share", ascending=False).to_string(index=False))
print("\nUnconstrained (diagnostic):")
print(gain_unconstrained_df.sort_values("mean_gain_share", ascending=False).to_string(index=False))

           feature  phase3_untuned  phase4_tuned_constrained  phase4_tuned_unconstrained
draft_pick_inverse        0.087252                  0.097371                    0.149747
               age        0.062606                  0.085022                    0.135632

Gap (draft_pick_inverse - age), each run:
  phase3_untuned: gap=+0.0246, draft_pick_inverse is 1.39x age
  phase4_tuned_constrained: gap=+0.0123, draft_pick_inverse is 1.15x age
  phase4_tuned_unconstrained: gap=+0.0141, draft_pick_inverse is 1.10x age

Full gain tables (all 4 features), for context:

Constrained (Phase 4 official):
           feature  mean_gain_share
        scarcity_z         0.756490
draft_pick_inverse         0.097371
               age         0.085022
    vorp_delta_yoy         0.061117

Unconstrained (diagnostic):
           feature  mean_gain_share
        scarcity_z         0.593783
draft_pick_inverse         0.149747
               age         0.135632
    vorp_delta_yoy         0.120838


## Naive baseline, same folds — apples-to-apples with Phase 3

In [8]:
naive_records = []
for tr, te, test_season, train_seasons in folds:
    actual = rb.loc[te, "vorp_next"]
    naive_pred = rb.loc[te, "vorp"]
    mae = mean_absolute_error(actual, naive_pred)
    rho = spearmanr(actual, naive_pred)[0] if len(actual) >= 2 and actual.nunique() > 1 else np.nan
    naive_records.append({"test_season": test_season, "mae": mae, "spearman": rho})
naive_df = pd.DataFrame(naive_records)
naive_mae, naive_spearman = naive_df["mae"].mean(), naive_df["spearman"].mean()
print(f"Naive baseline -- Mean MAE: {naive_mae:.2f}, Mean Spearman: {naive_spearman:.3f}")

Naive baseline -- Mean MAE: 47.97, Mean Spearman: 0.683


## Did tuning meaningfully close the gap Phase 3 found?

In [9]:
tuned_mae, tuned_spearman = fold_metrics_df["mae"].mean(), fold_metrics_df["spearman"].mean()
phase3_untuned_mae, phase3_untuned_spearman = 47.52, 0.691

comparison = pd.DataFrame([
    {"model": "Naive (this season's VORP)", "mae": round(naive_mae, 2), "spearman": round(naive_spearman, 3)},
    {"model": "Phase 3 untuned XGBoost", "mae": phase3_untuned_mae, "spearman": phase3_untuned_spearman},
    {"model": "Phase 4 tuned + monotonic (this notebook)", "mae": round(tuned_mae, 2), "spearman": round(tuned_spearman, 3)},
])
print(comparison.to_string(index=False))

mae_vs_naive_pct = (naive_mae - tuned_mae) / naive_mae
mae_vs_phase3_pct = (phase3_untuned_mae - tuned_mae) / phase3_untuned_mae
print(f"\nTuned vs naive: {mae_vs_naive_pct:.1%} MAE improvement, {tuned_spearman - naive_spearman:+.3f} Spearman")
print(f"Tuned vs Phase 3 untuned: {mae_vs_phase3_pct:+.1%} MAE change, {tuned_spearman - phase3_untuned_spearman:+.3f} Spearman change")

                                    model   mae  spearman
               Naive (this season's VORP) 47.97     0.683
                  Phase 3 untuned XGBoost 47.52     0.691
Phase 4 tuned + monotonic (this notebook) 46.67     0.691

Tuned vs naive: 2.7% MAE improvement, +0.008 Spearman
Tuned vs Phase 3 untuned: +1.8% MAE change, +0.000 Spearman change


## Save fold-by-fold results (overwrites Phase 3's untuned `fold_metrics_RB.csv`)

In [10]:
out_path = REPO_ROOT / "data" / "processed" / "fold_metrics_rb.csv"
fold_metrics_df.to_csv(out_path, index=False)
print(f"Saved {len(fold_metrics_df)} fold rows to {out_path}")

Saved 8 fold rows to C:\Users\viraj\OneDrive\Desktop\ML Trial\ML-Test\data\processed\fold_metrics_rb.csv


## Final model: trained on all available data, using the *stable* hyperparameter region

Not any single fold's exact best trial — a fold that happened to score best once can still reflect noise from a small test season. Instead, take the **median** of each hyperparameter across all 8 folds' best trials (rounded to valid integer values for `max_depth`/`min_child_weight`), which is far less sensitive to any one fold's idiosyncrasy.

`n_estimators` for the final model is found the same principled way real deployments do it: fit on all seasons through the second-to-last labeled one, early-stop against the most recent labeled season alone, then refit on **all** labeled data using that fixed iteration count — so the final artifact is trained on every real data point available, with no leftover held-out slice.

In [11]:
stable_params = {
    "max_depth": int(round(fold_params_df["max_depth"].median())),
    "min_child_weight": int(round(fold_params_df["min_child_weight"].median())),
    "reg_lambda": float(fold_params_df["reg_lambda"].median()),
    "learning_rate": float(fold_params_df["learning_rate"].median()),
    "subsample": float(fold_params_df["subsample"].median()),
}
print("Per-fold best hyperparameters:")
print(fold_params_df.drop(columns=["n_estimators"]).to_string(index=False))
print("\nStable (median) hyperparameters chosen for the final model:")
print(stable_params)

all_seasons_sorted = sorted(rb["season"].unique())
final_valid_season = all_seasons_sorted[-1]
final_train_seasons = all_seasons_sorted[:-1]

final_train_idx = rb.index[rb["season"].isin(final_train_seasons)]
final_valid_idx = rb.index[rb["season"] == final_valid_season]

X_final_train = rb.loc[final_train_idx, FEATURES]
y_final_train = rb.loc[final_train_idx, "vorp_next"]
X_final_valid = rb.loc[final_valid_idx, FEATURES]
y_final_valid = rb.loc[final_valid_idx, "vorp_next"]

probe_model = xgb.XGBRegressor(
    objective="reg:squarederror", eval_metric="mae", monotone_constraints=MONOTONE_CONSTRAINTS, seed=42,
    n_estimators=500, early_stopping_rounds=20, **stable_params,
)
probe_model.fit(X_final_train, y_final_train, eval_set=[(X_final_valid, y_final_valid)], verbose=False)
final_n_estimators = max(probe_model.best_iteration, 1)
print(f"\nFinal n_estimators (early-stopped against {final_valid_season}): {final_n_estimators}")

# Refit on ALL labeled data, no held-out slice left over, using the fixed iteration count.
final_model = xgb.XGBRegressor(
    objective="reg:squarederror", monotone_constraints=MONOTONE_CONSTRAINTS, seed=42,
    n_estimators=final_n_estimators, **stable_params,
)
final_model.fit(rb[FEATURES], rb["vorp_next"])

models_dir = REPO_ROOT / "data" / "models"
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / "rb_model.json"
final_model.save_model(str(model_path))
print(f"Final model trained on {len(rb)} rows (seasons {rb['season'].min()}-{rb['season'].max()}), saved to {model_path}")

Per-fold best hyperparameters:
 max_depth  min_child_weight  reg_lambda  learning_rate  subsample
         3                 8    0.249197       0.219572   0.682416
         2                 6    0.191596       0.269522   0.655309
         3                10    2.702067       0.288301   0.648991
         7                 6    2.664493       0.238783   0.613230
         3                 4    0.106956       0.287456   0.618955
         4                 3    0.166448       0.262993   0.653289
         4                 8    5.260729       0.160833   0.619029
         7                10    1.121330       0.242420   0.691861

Stable (median) hyperparameters chosen for the final model:
{'max_depth': 4, 'min_child_weight': 7, 'reg_lambda': 0.6852637738282819, 'learning_rate': 0.2527063312364141, 'subsample': 0.6511400682684466}

Final n_estimators (early-stopped against 2024): 17
Final model trained on 1823 rows (seasons 2008-2024), saved to C:\Users\viraj\OneDrive\Desktop\ML Trial\ML-T

## Does the final model still make football sense?

Checking specifically against Phase 3's finding: `draft_pick_inverse` outranked `age` in the untuned gain-importance chart. If tuning + monotonic constraints flipped that, that would be worth knowing before trusting this model's explanations later (Phase 5, SHAP).

In [12]:
gain_scores = final_model.get_booster().get_score(importance_type="gain")
importance_df = (
    pd.DataFrame({"feature": list(gain_scores.keys()), "gain": list(gain_scores.values())})
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)
# get_score() omits any feature never used in a split -- add those back in as 0 for a complete picture.
missing = [f for f in FEATURES if f not in importance_df["feature"].values]
if missing:
    importance_df = pd.concat([importance_df, pd.DataFrame({"feature": missing, "gain": 0.0})], ignore_index=True)
importance_df["gain_share"] = importance_df["gain"] / importance_df["gain"].sum()

print(importance_df.to_string(index=False))

rank = {f: i for i, f in enumerate(importance_df["feature"])}
draft_beats_age = rank.get("draft_pick_inverse", 99) < rank.get("age", 99)
print(f"\ndraft_pick_inverse outranks age: {draft_beats_age}")

           feature          gain  gain_share
        scarcity_z 173063.578125    0.750641
draft_pick_inverse  23883.988281    0.103594
               age  19795.777344    0.085862
    vorp_delta_yoy  13811.186523    0.059904

draft_pick_inverse outranks age: True


## Honest verdict

**Unlike QB, widening `max_depth` from 2-4 to 2-8 is a genuine win for RB on both metrics — checked directly rather than assumed to carry over.**

| Model | MAE | Spearman |
|---|---|---|
| Naive (this season's VORP) | 47.97 | 0.683 |
| Phase 3 untuned XGBoost | 47.52 | 0.691 |
| Phase 4 tuned + monotonic, max_depth 2-4 (previous, still shipped until now) | 47.42 | 0.681 |
| **Phase 4 tuned + monotonic, max_depth 2-8 (new, official)** | **46.67** | **0.691** |
| *(diagnostic: tuned, no constraints, max_depth 2-8)* | *47.90* | *0.691* |

This is a real, unambiguous improvement over the previously-shipped 2-4 model: MAE drops from 47.42 to 46.67 (a further 1.6% on top of the already-tuned model, 2.7% total over naive) and Spearman recovers from 0.681 to 0.691 — both metrics move the right direction simultaneously, not a mixed result. Because it wins on both MAE and Spearman, per the standing decision rule, **this run's model replaces the old 2-4 model as the shipped one** — `rb_model.json` and `fold_metrics_rb.csv` above are now built from the widened search.

**This does not mean wider trees are a universal win** — QB's own notebook found the opposite mix (a small MAE win alongside a Spearman give-back) using the identical change. RB's folds simply had more room to use here: the per-fold `max_depth` values chosen this run range from 2 to 7 (median 4), versus RB's old 2-4 search which could never explore past 4. Checking this per position, rather than assuming QB's finding carries over, was the right call.

**The monotonic-constraint diagnostic still goes the opposite way from QB's, and now more cleanly**: constraining `scarcity_z`/`vorp_delta_yoy` still *helps* MAE (46.67 vs. 47.90 unconstrained), and this time the old search's small Spearman cost to constraining has disappeared (0.691 vs. 0.691, an exact tie). RB's constrained model is now a clean win on MAE with no downside on ranking at all.

**Football-sense check, still no instability, unlike QB**: `scarcity_z` dominates even more than before (75.1% of gain vs. 80.2% under the old search — roughly the same overwhelming share). `draft_pick_inverse` still outranks `age` in the final model (10.4% vs. 8.6%), the same order Phase 3's untuned chart and every prior version of this notebook found. **This is the one clean contrast with QB's own re-run**: QB's final model saw its secondary-feature ranking flip (`passing_epa` over `draft_pick_inverse`) under the wider search; RB's ranking did not move at all.

**A reproducibility note, same as QB's notebook**: the exact numbers above shift slightly each time this notebook is re-executed, since the feature pipeline pulls live `nflreadpy` data rather than a frozen snapshot. The conclusions — constraints help RB's MAE (and no longer cost Spearman under the wider search), wider trees are a real win for RB (unlike QB, where they were a smaller, more mixed win), `scarcity_z` dominates, `draft_pick_inverse` outranks `age` — have held up across multiple runs and both `max_depth` ranges; the precise decimals have not.

**Bottom line**: RB gets a genuine, if still modest, edge from widening `max_depth` to 2-8 — a cleaner result than QB's own version of this same experiment, and specifically confirmed rather than assumed. The monotonic constraints stay locked in for RB (the opposite call from QB, made on RB's own held-out folds, not by analogy). `rb_model.json` and `fold_metrics_rb.csv` are updated to reflect this widened-search model as the new official one.

---

### Interface confirmation (the actual point of this exercise)

This notebook's `XGBRegressor`-first numbers, and QB's after being re-run through the same conversion, reproduced their pre-conversion `xgb.train()`/`DMatrix` results **exactly** (fold-by-fold MAE/RMSE/Spearman, per-fold best hyperparameters, final `n_estimators`, and gain-importance shares all matched to the displayed decimal places) — confirmed by executing both notebooks end-to-end and diffing outputs against the versions recorded before the conversion. This is the expected result: same seed, same params, same underlying booster call either way. Going forward, QB and RB (and any position built after them) share one interface: `XGBRegressor(...).fit(X, y, eval_set=[...], ...)`, `model.get_booster().get_score(...)` for gain importance, `model.best_iteration`/`model.best_score` for early stopping — no `DMatrix` construction anywhere.